<a href="https://colab.research.google.com/github/CryonicArtist/StochRSIAI/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import pandas as pd

# 1. Prompt authorization to connect your Drive
drive.mount('/content/drive')

# 2. Load the data (Update the path if you place it in a specific subfolder)
# If it is in the main directory of your Drive, this path will work:
file_path = '/content/drive/MyDrive/spy_1min_2008_2021_cleaned.csv'

df = pd.read_csv(file_path)
print("Data loaded successfully! Shape:", df.shape)

Mounted at /content/drive
Data loaded successfully! Shape: (2070834, 8)


In [ ]:
import pandas as pd
import numpy as np

# 1. Load SPY 1-minute CSV data
# Assuming file is in Colab environment or mounted Google Drive
file_path = 'spy_1min_2008_2021.csv'

df = pd.read_csv(
    file_path,
    parse_dates=['date'],
    index_col='date'
)
df.sort_index(inplace=True)

# 2. Translate OHLCV to Stationary Indicators for DRL
# Price Returns & Momentum
df['log_returns'] = np.log(df['close'] / df['close'].shift(1))

# Moving Averages & Relative Distances
df['sma_20'] = df['close'].rolling(window=20).mean()
df['sma_50'] = df['close'].rolling(window=50).mean()
df['dist_sma20'] = (df['close'] - df['sma_20']) / df['sma_20']
df['dist_sma50'] = (df['close'] - df['sma_50']) / df['sma_50']

# Relative Strength Index (RSI - 14 period)
delta = df['close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / (loss + 1e-8)
df['rsi_14'] = 100 - (100 / (1 + rs))

# Volatility: Normalized Average True Range (ATR) & Bollinger Bands %
high_low = df['high'] - df['low']
high_close = (df['high'] - df['close'].shift()).abs()
low_close = (df['low'] - df['close'].shift()).abs()
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df['natr_14'] = (tr.rolling(14).mean() / df['close']) * 100

std_20 = df['close'].rolling(20).std()
bb_upper = df['sma_20'] + (std_20 * 2)
bb_lower = df['sma_20'] - (std_20 * 2)
df['bb_pct'] = (df['close'] - bb_lower) / (bb_upper - bb_lower + 1e-8)

# Volume & Microstructure (Leveraging 'barCount' and 'average')
df['vol_sma20'] = df['volume'].rolling(20).mean()
df['volume_ratio'] = df['volume'] / (df['vol_sma20'] + 1e-8)
df['avg_trade_size'] = df['volume'] / (df['barCount'] + 1e-8)
df['vwap_dev'] = (df['close'] - df['average']) / df['average']

# Clean rolling NaNs
df.dropna(inplace=True)
print("Data shape after translation:", df.shape)
print(df[['log_returns', 'rsi_14', 'bb_pct', 'volume_ratio']].head())